In [15]:
import numpy as np
import pandas as pd
from pathlib import Path

# All Formula 1 CSVs (Ergast / Kaggle "F1 World Championship 1950-2020" schema)
# This notebook lives in the same folder as the CSVs, so load from the current dir.
DATA_DIR = Path(".")

dfs = {p.stem: pd.read_csv(p) for p in sorted(DATA_DIR.glob("*.csv"))}

# Expose each table as its own variable: circuits, drivers, races, results, ...
globals().update(dfs)

for name, df in dfs.items():
    print(f"{name:<22} {df.shape[0]:>7} rows x {df.shape[1]:>2} cols")

circuits                    78 rows x  9 cols
constructor_results      12942 rows x  5 cols
constructor_standings    13708 rows x  7 cols
constructors               214 rows x  5 cols
driver_standings         35515 rows x  7 cols
drivers                    865 rows x  9 cols
lap_times               873755 rows x  6 cols
pit_stops                22383 rows x  7 cols
qualifying               11124 rows x  9 cols
races                     1171 rows x 18 cols
results                  27392 rows x 18 cols
seasons                     77 rows x  2 cols
sprint_results             546 rows x 17 cols
status                     140 rows x  2 cols


In [16]:
# Read a single file into a pandas DataFrame
df = pd.read_csv("results.csv")

# 1) How many rows and columns? -> (rows, columns)
print("Shape (rows, columns):", df.shape)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Shape (rows, columns): (27392, 18)
Number of rows: 27392
Number of columns: 18


In [17]:
# 2) What columns are there?
print("Columns:")
print(list(df.columns))

Columns:
['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId']


In [18]:
# 3) What data type is each column? (object = text/string in pandas)
print(df.dtypes)

resultId             int64
raceId               int64
driverId             int64
constructorId        int64
number              object
grid                object
position            object
positionText        object
positionOrder        int64
points             float64
laps                 int64
time                object
milliseconds        object
fastestLap          object
rank                object
fastestLapTime      object
fastestLapSpeed     object
statusId             int64
dtype: object


In [19]:
# 4) Everything at once: rows, columns, dtypes, and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27392 entries, 0 to 27391
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   resultId         27392 non-null  int64  
 1   raceId           27392 non-null  int64  
 2   driverId         27392 non-null  int64  
 3   constructorId    27392 non-null  int64  
 4   number           27392 non-null  object 
 5   grid             27392 non-null  object 
 6   position         27392 non-null  object 
 7   positionText     27392 non-null  object 
 8   positionOrder    27392 non-null  int64  
 9   points           27392 non-null  float64
 10  laps             27392 non-null  int64  
 11  time             27392 non-null  object 
 12  milliseconds     27392 non-null  object 
 13  fastestLap       27392 non-null  object 
 14  rank             27392 non-null  object 
 15  fastestLapTime   27392 non-null  object 
 16  fastestLapSpeed  27392 non-null  object 
 17  statusId    

In [20]:
# 5) Peek at the first 5 rows to see what the data actually looks like
df.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.3,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1


## 6) Merging tables — show driver *names*, not just `driverId`

`results` stores a numeric `driverId` instead of a name (to avoid repeating
"Lewis Hamilton" thousands of times). The `drivers` table is the lookup that
maps each `driverId` -> `forename`, `surname`, etc.

**Merging** = joining two tables on a shared key column. Here the key is
`driverId`, which exists in both tables.

- `on="driverId"` — the shared column to match rows by
- `how="left"`   — keep every row from the left table (`results`) and attach
  driver info where it matches (so we never lose a race result)

In [22]:
# Grab only the columns we need from the drivers lookup table
driver_names = drivers[["driverId", "forename", "surname"]]

# Join driver names onto every result row, matching on driverId
merged = results.merge(driver_names, on="driverId", how="left")

# Combine forename + surname into a single readable "driver" column
merged["driver"] = merged["forename"] + " " + merged["surname"]

# Show results with the name instead of just the ID
merged[[["resultId", "raceId", 'driverId', 'driver', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId']]].head()

KeyError: "None of [Index([('resultId', 'raceId', 'driverId', 'driver', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId')], dtype='object')] are in the [columns]"

In [ ]:
# Same pattern, chained: each extra table is just another .merge() on its ID
out = (
    results
    .merge(drivers[["driverId", "forename", "surname"]], on="driverId", how="left")
    .merge(constructors[["constructorId", "name"]], on="constructorId", how="left")
    .merge(races[["raceId", "name", "year"]], on="raceId", how="left",
           suffixes=("_constructor", "_race"))  # both tables have a "name" column
)

out["driver"] = out["forename"] + " " + out["surname"]
out = out.rename(columns={"name_constructor": "team", "name_race": "grand_prix"})

out[["year", "grand_prix", "driver", "team", "grid", "position", "points"]].head()

In [ ]:
df.head()